# Fine-tune the Annotrix detector on a GPU (university Data Lab JupyterHub)

The stock COCO detector has no **buffalo** or **rhino** class (AP 0.00 on the African Wildlife test split).
This notebook fine-tunes YOLO11 on the African Wildlife training split, evaluates it with the same
script as the report, and gives you the weights to drop into `data/weights/` of the app.

Run the cells top to bottom. Expected time on one modern GPU: ~15–30 min for 60 epochs of `yolo11s`.

In [ ]:
# 1. Check the GPU
!nvidia-smi
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. Get the code (replace with your repository URL)
REPO = "https://github.com/<your-username>/annotrix.git"
!git clone -q $REPO annotrix || (cd annotrix && git pull -q)
%cd annotrix
!pip install -q -r requirements-eval.txt

In [ ]:
# 3. Download African Wildlife (+ COCO128-seg) and build the ground-truth files
!python scripts/prepare_data.py

In [ ]:
# 4. Fine-tune (COCO-pretrained yolo11s as the starting point)
!python scripts/finetune.py --model yolo11s.pt --epochs 60 --batch 32 --device 0

In [ ]:
# 5. Evaluate stock vs fine-tuned on the test split (same metrics as the report)
!python eval/eval_detector.py --weights yolo11s.pt data/weights/wildlife-yolo11s.pt

In [ ]:
# 6. Look at the training curves and the failure gallery
from IPython.display import Image, display, Markdown
display(Markdown(open("eval/results/detector/summary.md").read()))
display(Image("runs/wildlife-yolo11s/results.png", width=900))
display(Image("eval/results/detector/failures_wildlife-yolo11s.png", width=900))

In [ ]:
# 7. Optional: segmenter evaluation with a larger SAM on the GPU
!python eval/eval_segmenter.py --weights sam2.1_b.pt --max-objects 400 --clicks 5

## Use the weights in the app
Download `data/weights/wildlife-yolo11s.pt` (right-click → Download in the Jupyter file browser), put it in
`data/weights/` of your local checkout, then start the app with

```
LA_DETECTOR=wildlife-yolo11s.pt docker compose up
```

Commit `eval/results/` so the report's numbers are reproducible (weights are git-ignored — upload them as a release asset if you want to share them).